### Prompt Evaluation
#### Goal: After you develop a prompt, use criteria to determine how well it is working

1. Draft prompt (this is what you will do A/B testing on)
2. Create an evaluation dataset
    - "Contains sample inputs that represent the types of questions or requests your prompt will handle in production. The dataset should include questions that will be interpolated into your prompt template"
3. Send to Claude or LMM to get response
4. Send to Grader
    - "The grader evaluates the quality of Claude's responses by examining both the original question and Claude's answer. This step provides objective scoring, typically on a scale from 1 to 10, where 10 represents a perfect answer and lower scores indicate room for improvement."
5. Change prompt and repeat


**Note:** The evaluation dataset doesn't change, it is the prompt that is changed and what are you A/B testing


In [7]:
# Load env variables and create client
import json
import logging
import os
import sys
import re
import ast

from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic(api_key=os.getenv("CLAUDE_COURSE_API_KEY"))
model = "claude-sonnet-4-0"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

#### Generate data evaluation dataset as dict

In [3]:
# Function to generate a new dataset

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [4]:
# Generate the dataset and write it to 'dataset.json'
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [14]:
dataset

[{'task': "Create a JSON policy document that allows an IAM user to read objects from a specific S3 bucket named 'my-app-logs' and list the bucket contents",
  'format': 'json',
  'solution_criteria': "Must include correct IAM policy structure with Version, Statement array, Effect 'Allow', Principal or not specified for user attachment, Action array with s3:GetObject and s3:ListBucket, and Resource ARNs for the specific bucket and its objects"},
 {'task': 'Write a Python function that takes an EC2 instance ID and returns a properly formatted CloudWatch metric filter name following AWS naming conventions (alphanumeric and hyphens only)',
  'format': 'python',
  'solution_criteria': 'Function should accept instance ID parameter, replace any invalid characters with hyphens, ensure no consecutive hyphens, strip leading/trailing hyphens, and return a string suitable for CloudWatch metric filter naming'},
 {'task': 'Create a regular expression that validates AWS Lambda function names accordi

### Graders
#### Code Graders
- Code graders let you implement any programmatic check you can imagine. Common uses include:
    - Checking output length
    - Verifying output does/doesn't have certain words
    - Syntax validation for JSON, Python, or regex
    - Readability scores
- The only requirement is that your code returns some usable signal - usually a number between 1 and 10.

#### Model Graders
- Model graders feed your original output into another API call for evaluation. This approach offers tremendous flexibility for assessing:
    - Response quality
    - Quality of instruction following
    - Completeness
    - Helpfulness
    - Safety


#### Human Graders
- Human graders provide the most flexibility but are time-consuming and tedious. They're useful for evaluating:
    - General response quality
    - Comprehensiveness
    - Depth
    - Conciseness
    - Relevance

In [5]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [6]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [8]:
# Functions to validate the output structure

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [9]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [10]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [11]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 7.666666666666667


In [12]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n{\n  \"Version\": \"2012-10-17\",\n  \"Statement\": [\n    {\n      \"Effect\": \"Allow\",\n      \"Action\": [\n        \"s3:GetObject\"\n      ],\n      \"Resource\": \"arn:aws:s3:::my-app-logs/*\"\n    },\n    {\n      \"Effect\": \"Allow\",\n      \"Action\": [\n        \"s3:ListBucket\"\n      ],\n      \"Resource\": \"arn:aws:s3:::my-app-logs\"\n    }\n  ]\n}\n",
    "test_case": {
      "task": "Create a JSON policy document that allows an IAM user to read objects from a specific S3 bucket named 'my-app-logs' and list the bucket contents",
      "format": "json",
      "solution_criteria": "Must include correct IAM policy structure with Version, Statement array, Effect 'Allow', Principal or not specified for user attachment, Action array with s3:GetObject and s3:ListBucket, and Resource ARNs for the specific bucket and its objects"
    },
    "score": 9.5,
    "reasoning": "The policy correctly implements the core requirements with proper JSON structure, a